# Dados agrícolas do IBGE para o estudo de crédito rural

Este notebook prepara dados anuais da Produção Agrícola Municipal (PAM), tabela SIDRA 1612, para soja em grão, milho em grão e cana-de-açúcar. O código atual prevê o período de 2016 a 2025. 

As perguntas de referência são:

| Referência | Pergunta de domínio |
|---|---|
| P1 | Como evoluíram o valor contratado e o número de contratos de custeio por cultura na década? |
| P2 | Como o crédito e a área plantada se distribuem entre estados e regiões? |
| P3 | O crédito está mais concentrado em poucos municípios do que a produção? |
| P4 | Como variam o crédito e os contratos entre municípios com áreas plantadas semelhantes? |
| P5 | Como variam o crédito por tonelada e por hectare entre municípios e estados? |
| P6 | Municípios com maior crédito por hectare apresentam distribuições diferentes de produtividade, na mesma cultura e ano? |
| P7 | Quais municípios apresentam valores atípicos de crédito relativo ou produtividade no padrão dos estados selecionados? |

### Tratamento dos valores e cobertura

O código converte `-` em zero e transforma os valores em pontos flutuantes caso sejam dígitos. Os demais registros são descartados, pois na convenção do IBGE, `..` indica não aplicável; `...`, indisponível; e `X`, informação omitida, informações que não irão agregar nas análises.

### 1. Bibliotecas de trabalho

Importa as bibliotecas usadas pelo notebook. `pandas` organiza as respostas em tabelas e `requests` realiza as consultas HTTP.

In [9]:
import pandas as pd
import requests as rqs
from urllib.parse import quote, urlencode
import hashlib
import json
from functools import lru_cache
from datetime import datetime, timezone

### 2. Validação do certificado HTTPS

Define o adaptador `IBGETLSAdapter`, que complementa a cadeia de certificados do IBGE com o intermediário local.

**Por que é necessária:** permite consultar o serviço com validação ativa considerando a cadeia incompleta de certifiação identificada no servidor.

In [10]:
import ssl
from pathlib import Path
from requests.adapters import HTTPAdapter


class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE sem desativar a valida??o TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermediário deve encadear até uma raiz confiável, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs['ssl_context'] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs['ssl_context'] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)

### 3. Identificadores das culturas e variáveis

Declaração de variáveis dos códigos de soja, milho e cana-de-açúcar, da tabela 1612, das variáveis agrícolas e do certificado. Área plantada (`109`) e colhida (`216`) usam hectares; produção (`214`), toneladas; rendimento (`112`), kg/ha.

In [11]:
BASE = "https://servicodados.ibge.gov.br/api/v3/agregados/"
CACHE = Path("dados/ibge/cache")
id_milho = 2711
id_soja = 2713
id_cana = 2696
#Área plantada Hectares
area_plant_H = 109
#Área plantada - percentual do total geral %
area_plant_100 = 1000109
#Área colhida Hectares
area_colhida_H = 216
#Área colhida - percentual do total geral %
area_colhida_100 = 1000216
#Quantidade produzida Toneladas
quant_prod_T = 214
#Rendimento médio da produção Quilogramas por Hectare
rend_medio_Kg_H = 112
#Valor da produção - percentual do total geral %
tabela_lavouras_temporárias = 1612

parametros = {
    "localidades": "N2[all]",
    "classificacao": f"81[{id_soja},{id_cana},{id_milho}]"
}

certificado_intermediario = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")

In [12]:
def _endereco_consulta(url):
    """Monte o caminho de cache usados por todas as consultas."""
    return CACHE / (hashlib.sha256(url.encode()).hexdigest() + ".json")

def _salvar_cache(arquivo, registro):
    """Grave o registro completo antes de substituir o arquivo de cache."""
    CACHE.mkdir(parents=True, exist_ok=True)
    temporario = arquivo.with_suffix(".tmp")
    temporario.write_text(json.dumps(registro, ensure_ascii=False), encoding="utf-8")
    temporario.replace(arquivo)
    
@lru_cache(maxsize=16)
def _ler_cache_json(arquivo, modificado_em_ns, tamanho):
    """Reutilize o JSON em memória enquanto a versão do arquivo for a mesma.

    O caminho absoluto, a data de modificação e o tamanho identificam
    a versão. O limite evita manter todo o histórico em memória.
    """
    return json.loads(arquivo.read_text(encoding="utf-8"))

def requisicao_certificada(url: str, parametros: dict = {}) -> dict:
    with rqs.Session() as sessao:
        sessao.mount("https://servicodados.ibge.gov.br/", IBGETLSAdapter(certificado_intermediario))
        try:
            response = sessao.get(url,params=parametros, timeout=30)
            response.raise_for_status()
        except rqs.RequestException as erro:
            status = erro.response.status_code if erro.response is not None else None
            temporaria = (
                status in (429, 500, 502, 503, 504)
                or isinstance(erro, (rqs.Timeout, rqs.ConnectionError))
                and not isinstance(erro, rqs.exceptions.SSLError)
            )
            erro = Exception if temporaria else RuntimeError
            raise erro(f"Falha ao consultar BCB: {url}\n{erro}") from erro
        return response.json()

def consultar(url,parametros, *, limite=5, completo=False, atualizar=False) -> dict:
    """Consulte um recurso OData com cache local e metadados de auditoria.

    Parameters
    ----------
    limite : int, optional
        Máximo de registros solicitado, de 1 a ``LIMITE_LOTE``. Padrão: 5.
    completo : bool, optional
        Se True, rejeita respostas com indício de truncamento. Padrão: False.
    atualizar : bool, optional
        Se True, consulta a API e substitui o cache existente. Padrão: False.

    Returns
    -------
    pandas.DataFrame
        Registros retornados, possivelmente vazios. Os atributos ``attrs``
        incluem URL, instante UTC da obtenção, opção de consulta completa
        e caminho do arquivo de cache.

    Raises
    ------
    ValueError
        Se o recurso, limite, JSON ou lista de registros for inválido,
        ou se a URL armazenada no cache não corresponder à consulta.
    RuntimeError
        Se a requisição falhar ou houver possível truncamento com
        ``completo=True``.
    OSError
        Se ocorrer falha ao ler ou gravar arquivos de cache.

    Notes
    -----
    O cache usa o hash SHA-256 da URL e não expira automaticamente.
    Os últimos 16 JSONs lidos são reutilizados em memória enquanto o
    arquivo não mudar. ``atualizar=True`` limpa esse reaproveitamento.
    Não há paginação: atingir o limite ou receber ``@odata.nextLink``
    é considerado indício de truncamento quando ``completo=True``.
    """
    
    arquivo = _endereco_consulta(url)
    if arquivo.exists() and not atualizar:
        estado = arquivo.stat()
        registro = _ler_cache_json(arquivo.resolve(), estado.st_mtime_ns, estado.st_size)
        if registro["url"] != url:
            raise ValueError("Cache incompatível com a consulta.")
    else:
        if atualizar:
            _ler_cache_json.cache_clear()
        try:
            dados = requisicao_certificada(url,parametros)
        except ValueError as erro:
            raise ValueError("O BCB não retornou JSON válido.") from erro
        if not isinstance(dados, dict) or not isinstance(dados.get("value"), list):
            raise ValueError("Resposta OData sem a lista 'value'.")
        registro = {
            "url": url, "obtido_em_utc": datetime.now(timezone.utc).isoformat(),
            "dados": dados, "consulta_completa": completo, "arquivo_cache": str(arquivo)
        }
        _salvar_cache(arquivo, registro)
    dados = registro["dados"]
    if not isinstance(dados.get("value"), list):
        raise ValueError("Cache inválido: lista 'value' ausente.")
    if completo and (len(dados["value"]) >= limite or dados.get("@odata.nextLink")):
        raise Exception("Possível truncamento. Reduza o recorte; não use $skip.")
    return dados

### 4. Função de requisição certificada

Encapsula a abertura da sessão HTTPS, o envio dos parâmetros, o limite de espera, a verificação do status HTTP e a leitura do JSON.

In [13]:
def criar_dataframe(lista: list, dados: dict, localizador: str) -> list:
    variavel = dados[0]
    resultado = variavel["resultados"][0]

    for resultado in variavel["resultados"]:
        categorias = resultado["classificacoes"][0]["categoria"]

        for codigo_cultura, nome_cultura in categorias.items():
            for serie in resultado["series"]:
                for ano, valor in serie["serie"].items():
                    if valor == "-":
                        valor = 0
                    elif not valor.isdecimal():
                        continue
                    lista.append({
                        "codigo_cultura": codigo_cultura,
                        "cultura": nome_cultura,
                        f"codigo_{localizador}": serie["localidade"]["id"],
                        localizador: serie["localidade"]["nome"],
                        "ano": int(ano),
                        "variavel": variavel["variavel"],
                        "unidade": variavel["unidade"],
                        "valor": float(valor)
                    })
    return lista

### 5. Conferência dos metadados da tabela

Consulta os nomes, códigos e unidades das variáveis e as categorias da classificação de produtos das lavouras temporárias.

In [14]:

url_metadados = f"https://servicodados.ibge.gov.br/api/v3/agregados/{tabela_lavouras_temporárias}/metadados"

dados = requisicao_certificada(url_metadados)

print("Tabela:", dados["nome"])

for variavel in dados["variaveis"]:
    print(
        variavel["id"],
        variavel["nome"],
        variavel["unidade"]
    )

for classificacao in dados["classificacoes"]:
    print("Classificação:", classificacao["id"], classificacao["nome"])

    for categoria in classificacao["categorias"]:
        print(categoria["id"], categoria["nome"])

for classificacao in dados["classificacoes"]:
    print(classificacao["id"], classificacao["nome"])

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %
Classificação: 81 Produto das lavouras temporárias
0 Total
2688 Abacaxi*
83374 Abóbora
40471 Alfafa fenada
83375 Alface
2689 Algodão herbáceo (em caroço)
2690 Alho
2691 Amendoim (em casca)
2692 Arroz (em casca)
2693 Aveia (em grão)
2694 Batata-doce
2695 Batata-inglesa
2696 Cana-de-açúcar
40470 Cana para forragem
83384 Canola
2697 Cebola
83381 Cenoura
2698 Centeio (em grão)
2699 Cevada (em grão)
83379 

### 6. Área plantada por região

Solicita área plantada em hectares (`109`), para as três culturas, nas Grandes Regiões (`N2`), entre 2016 e 2025, e pretende organizar o resultado em `df_area_regional`.

**Pergunta apoiada:** P2, na parte de área plantada.

In [15]:
periodo = "2016-2025"


url_area_regiao = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_H}"
)

dados = requisicao_certificada(url_area_regiao, parametros=parametros)

linhas_area_regional = []

linhas_area_regional = criar_dataframe(linhas_area_regional, dados, "regiao")

df_area_regional = pd.DataFrame(linhas_area_regional)
df_area_regional = df_area_regional.sort_values(["regiao","cultura","ano","valor"])

df_area_regional.head()

,codigo_cultura,cultura,codigo_regiao,regiao,ano,variavel,unidade,valor
90,2696,Cana-de-açúcar,5,Centro-Oeste,2016,Área plantada,Hectares,1870011.0
91,2696,Cana-de-açúcar,5,Centro-Oeste,2017,Área plantada,Hectares,1864280.0
92,2696,Cana-de-açúcar,5,Centro-Oeste,2018,Área plantada,Hectares,1914387.0
93,2696,Cana-de-açúcar,5,Centro-Oeste,2019,Área plantada,Hectares,1972043.0
94,2696,Cana-de-açúcar,5,Centro-Oeste,2020,Área plantada,Hectares,1859847.0


### 7. Área plantada por estado

Altera o nível territorial para Unidades da Federação (`N3`) e pretende obter a área plantada das três culturas no mesmo intervalo regional, formando `df_area_estadual`.

**Perguntas apoiadas:** P2 e P5, na parte agrícola.

In [16]:
parametros["localidades"] = "N3[all]"

dados = requisicao_certificada(url_area_regiao, parametros=parametros)

linhas_area_estadual = []

linhas_area_regional = criar_dataframe(linhas_area_estadual, dados, "estado")
            

df_area_estadual = pd.DataFrame(linhas_area_estadual)
df_area_estadual = df_area_estadual.sort_values(["estado","cultura","ano","valor"])

df_area_estadual.head()

,codigo_cultura,cultura,codigo_estado,estado,ano,variavel,unidade,valor
280,2696,Cana-de-açúcar,12,Acre,2016,Área plantada,Hectares,3079.0
281,2696,Cana-de-açúcar,12,Acre,2017,Área plantada,Hectares,2437.0
282,2696,Cana-de-açúcar,12,Acre,2018,Área plantada,Hectares,517.0
283,2696,Cana-de-açúcar,12,Acre,2019,Área plantada,Hectares,513.0
284,2696,Cana-de-açúcar,12,Acre,2020,Área plantada,Hectares,490.0


### 8. Quantidade produzida por município

Solicita produção em toneladas (`214`) nos municípios (`N6`), ano a ano de 2016 a 2025, para as três culturas, e reúne os registros em `df_prod_municipal`.

**Perguntas apoiadas:** P3 e P5; também auxilia a conferência agrícola de P7.

In [17]:
parametros["localidades"] = "N6[all]"
linhas_quant_municipal = []
for periodo in range(2016,2026):
    url_quant_municipio = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
    )

    dados = requisicao_certificada(url_quant_municipio, parametros=parametros)

    linhas_quant_municipal = criar_dataframe(linhas_quant_municipal, dados, "municipio")
                

df_prod_municipal = pd.DataFrame(linhas_quant_municipal)
df_prod_municipal = df_prod_municipal.sort_values(["municipio","cultura","ano","valor"])

df_prod_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Quantidade produzida,Toneladas,0.0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Quantidade produzida,Toneladas,300.0
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Quantidade produzida,Toneladas,238.0
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Quantidade produzida,Toneladas,1208.0
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Quantidade produzida,Toneladas,0.0


### 9. Quantidade produzida por estado

Solicita produção em toneladas (`214`) por Unidade da Federação (`N3`), de 2016 a 2025, para as culturas selecionadas.

**Pergunta apoiada:** P5, na parte agrícola estadual.

In [18]:
parametros["localidades"] = "N3[all]"
linhas_quant_estadual = []
periodo = "2016-2025"
url_quant_estado = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
)

dados = requisicao_certificada(url_quant_estado, parametros=parametros)

linhas_quant_estadual = criar_dataframe(linhas_quant_estadual, dados, "estado")
                

df_prod_estadual = pd.DataFrame(linhas_quant_estadual)
df_prod_estadual = df_prod_estadual.sort_values(["estado","cultura","ano","valor"])

df_prod_estadual.head()

,codigo_cultura,cultura,codigo_estado,estado,ano,variavel,unidade,valor
280,2696,Cana-de-açúcar,12,Acre,2016,Quantidade produzida,Toneladas,117244.0
281,2696,Cana-de-açúcar,12,Acre,2017,Quantidade produzida,Toneladas,20229.0
282,2696,Cana-de-açúcar,12,Acre,2018,Quantidade produzida,Toneladas,11989.0
283,2696,Cana-de-açúcar,12,Acre,2019,Quantidade produzida,Toneladas,12119.0
284,2696,Cana-de-açúcar,12,Acre,2020,Quantidade produzida,Toneladas,11822.0


### 10. Área plantada por município

Solicita área plantada em hectares (`109`) nos municípios (`N6`), para as três culturas e cada ano de 2016 a 2025, formando `df_area_municipal`.

**Perguntas apoiadas:** P4 e P5; auxilia a identificação de denominadores pequenos em P7.

In [19]:
parametros["localidades"] = "N6[all]"
linhas_area_municipal = []
for periodo in range(2016,2026):
    url_area_municipio = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_H}"
    )

    dados = requisicao_certificada(url_area_municipio, parametros=parametros)

    linhas_area_municipal = criar_dataframe(linhas_area_municipal, dados, "municipio")
                    

df_area_municipal = pd.DataFrame(linhas_area_municipal)

df_area_municipal = df_area_municipal.sort_values(["municipio","cultura","ano","valor"])
df_area_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Área plantada,Hectares,0.0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Área plantada,Hectares,5.0
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Área plantada,Hectares,4.0
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Área plantada,Hectares,15.0
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Área plantada,Hectares,0.0


### 11. Rendimento médio por município

Solicita o rendimento médio oficial (`112`), em kg/ha, para as três culturas, nos municípios (`N6`), de 2016 a 2025, formando `df_rendimento_municipal`.

**Perguntas apoiadas:** P6 e P7, na parte agrícola.

In [20]:
parametros["localidades"] = "N6[all]"
linhas_rendimento = []
for periodo in range(2016,2026):
    url_rend_municipio = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{rend_medio_Kg_H}"
    )

    dados = requisicao_certificada(url_rend_municipio, parametros=parametros)

    linhas_rendimento = criar_dataframe(linhas_rendimento, dados, "municipio")

df_rendimento_municipal = pd.DataFrame(linhas_rendimento)

df_rendimento_municipal = df_rendimento_municipal.sort_values(["municipio","cultura","ano","valor"])
df_rendimento_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Rendimento médio da produção,Quilogramas por Hectare,0.0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Rendimento médio da produção,Quilogramas por Hectare,60000.0
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Rendimento médio da produção,Quilogramas por Hectare,59500.0
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Rendimento médio da produção,Quilogramas por Hectare,80533.0
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Rendimento médio da produção,Quilogramas por Hectare,0.0
